# SIH26012 — AI-Based Automated Urban Parcel Mapping
## Phase 4: Ultra-Fast GPU Sanity-Training Run (<10 Minutes Verification)
### Progressive Multiscale Generator (PMG) + Domain Generalization (DG) + Connectivity Dual-Head + Residual U-Net

This notebook executes the ultra-fast sanity run (**50 train, 10 val, 10 test patches @ 256×256, 1 epoch**) to verify end-to-end:
- Data loading and 256×256 preprocessing
- Forward pass across the full PMG + DG + Connectivity architecture
- Multi-task gradient descent with mixed precision (AMP)
- Validation loop and metric computation (F1, IoU, Precision, Recall, Dice)
- Model checkpointing (`best_model.pth`, `latest_checkpoint.pth`)
- Final test evaluation on held-out test patches and GIS vectorization

---

### Step 1: Verify Hardware Accelerator (Colab GPU)

In [ ]:
!pip install -q rasterio geopandas shapely pyogrio opencv-python-headless matplotlib pillow pyyaml tqdm networkx tabulate uvicorn fastapi

import os, sys, time, json, random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

print("=== Hardware Accelerator Inspection ===")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Active GPU:      {gpu_name}")
    print(f"Total VRAM:      {vram_gb:.2f} GB")
    torch.backends.cudnn.benchmark = True
else:
    print("WARNING: CUDA not detected! Please go to Runtime -> Change runtime type -> Select GPU.")

### Step 2: Mount Google Drive & Set Working Directory
*(Skip this cell if you already ran `git clone` or are running in the repository root directory)*

In [ ]:
if os.path.exists('/content'):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        drive_dir = '/content/drive/MyDrive/SIH26012_COLAB'
        if os.path.exists(drive_dir):
            os.chdir(drive_dir)
            print(f"[✓] Switched working directory to: {os.getcwd()}")
    except Exception as e:
        print(f"Drive mount note: {e}")

print("Current Working Directory:", os.getcwd())

### Step 3: Inspect Sanity Configuration & Verify Dataset Catalogs

In [ ]:
import yaml
config_path = "configs/sanity_full.yaml"
assert os.path.exists(config_path), f"Missing {config_path}!"

with open(config_path, "r") as f:
    sanity_cfg = yaml.safe_load(f)

print("=== Sanity Configuration ===")
print(f"Architecture:      {sanity_cfg['model']['architecture']}")
print(f"PMG Enabled:       {sanity_cfg['model']['pmg']['enabled']}")
print(f"DG Enabled:        {sanity_cfg['model']['domain_generalization']['enabled']}")
print(f"Connectivity:      {sanity_cfg['model']['connectivity']['enabled']}")
print(f"Train Samples:     {sanity_cfg['data']['max_train_samples']}")
print(f"Val Samples:       {sanity_cfg['data']['max_val_samples']}")
print(f"Test Samples:      {sanity_cfg['data']['max_test_samples']}")
print(f"Image Size:        {sanity_cfg['data']['image_size']}")
print(f"Epochs:            {sanity_cfg['training']['epochs']}")
print(f"Batch Size:        {sanity_cfg['training']['batch_size']}")

### Step 4: Execute Ultra-Fast Full-Architecture Sanity Run
Runs 1 epoch on 50 train patches, evaluates on 10 val patches, saves checkpoints, and evaluates on 10 test patches.

In [ ]:
# Execute sanity run with full PMG + DG + Connectivity architecture
!python scripts/train_full.py configs/sanity_full.yaml

### Step 5: Check Saved Sanity Metrics & Artifacts

In [ ]:
history_path = "experiments/sanity_full/metrics/training_history.json"
test_metrics_path = "experiments/sanity_full/metrics/metrics_test.json"
ckpt_path = "experiments/sanity_full/checkpoints/best_model.pth"

assert os.path.exists(ckpt_path), f"Checkpoint missing: {ckpt_path}"
print(f"[✓] Best model checkpoint successfully saved: {ckpt_path} ({os.path.getsize(ckpt_path)/1e6:.2f} MB)")

if os.path.exists(history_path):
    with open(history_path) as f:
        history = json.load(f)
    print("\n=== Training History ===")
    df_hist = pd.DataFrame(history)
    display(df_hist)

if os.path.exists(test_metrics_path):
    with open(test_metrics_path) as f:
        test_m = json.load(f)
    print("\n=== Sanity Test Set Metrics (10 Patches) ===")
    print(f"Test F1 Score:   {test_m['f1']:.4f}")
    print(f"Test IoU:        {test_m['iou']:.4f}")
    print(f"Test Precision:  {test_m['precision']:.4f}")
    print(f"Test Recall:     {test_m['recall']:.4f}")

### Step 6: Verify Sample Prediction Visualizations (5-Panel)

In [ ]:
import glob
from PIL import Image

vis_files = sorted(glob.glob("experiments/sanity_full/predictions/test_vis_*.png"))
print(f"[✓] Found {len(vis_files)} prediction visualization figures.")

if len(vis_files) > 0:
    img = Image.open(vis_files[0])
    plt.figure(figsize=(16, 5), dpi=120)
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"Sanity Test Sample Visualization: {os.path.basename(vis_files[0])}")
    plt.show()

### Step 7: End-to-End GIS Raster-to-Vector & Topology Audit Verification

In [ ]:
from ai.models.full_model import CadastreUNetFull
from gis.pipeline import RasterToVectorGISPipeline

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CadastreUNetFull().to(device)
ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

df_test = pd.read_csv("data/processed/metadata/patches_test.csv")
sample_row = df_test.iloc[0]
img_pil = Image.open(os.path.join("data/processed", sample_row["image_path"])).convert("RGB").resize((256, 256))
img_tensor = torch.from_numpy(np.array(img_pil).transpose(2, 0, 1) / 255.0).unsqueeze(0).float().to(device)

with torch.no_grad():
    pred_out = model(img_tensor, return_dict=True)
    prob_map = pred_out["refined_prob"][0, 0].cpu().numpy()

pipeline = RasterToVectorGISPipeline(threshold=0.5, simplify_tolerance=0.5)
gis_result = pipeline.run(prob_map, transform=(0.25, 0.0, 100000.0, 0.0, -0.25, 500000.0))

print("=== GIS Vectorization & Topology Audit Report ===")
print(f"Total Vector Lines:      {gis_result['total_lines']}")
print(f"Total Boundary Length:   {gis_result['total_length_m']} m")
print(f"Topology Status:         {gis_result['topology_report']['status']}")
print(f"Self-Intersections:      {gis_result['topology_report']['self_intersections_count']}")
print(f"Invalid Geometries:      {gis_result['topology_report']['invalid_geometries_count']}")
print("\n[✓] ULTRA-FAST SANITY RUN FULLY VERIFIED! Ready for production full-scale training.")